# 📊 FinanceTool Data Viewer

**Purpose**: Interactive viewer for all generated data, models, and visualizations

**Last Updated**: 2026-02-12

---

## Quick Navigation
1. [Data Summary](#data-summary)
2. [Model Information](#model-information)
3. [Visualizations](#visualizations)
4. [Performance Metrics](#performance-metrics)
5. [Raw Data Exploration](#raw-data-exploration)

In [ ]:
# Setup and imports
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display, HTML
import joblib
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Set styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All imports successful")
print(f"📁 Project root: {project_root}")

---
## 📈 Data Summary

In [ ]:
# Scan for available data files
data_dir = project_root / "data"
models_dir = project_root / "models"
figures_dir = project_root / "docs" / "figures"

# Count files
raw_files = list((data_dir / "raw").glob("*.csv")) if (data_dir / "raw").exists() else []
processed_files = list((data_dir / "processed").glob("*.csv")) if (data_dir / "processed").exists() else []
model_files = list(models_dir.glob("*.pkl")) if models_dir.exists() else []
figure_files = list(figures_dir.glob("*.png")) if figures_dir.exists() else []

# Create summary DataFrame
summary_data = {
    "Category": ["Raw Data Files", "Processed Data Files", "Trained Models", "Visualizations"],
    "Count": [len(raw_files), len(processed_files), len(model_files), len(figure_files)],
    "Location": [
        "data/raw/",
        "data/processed/",
        "models/",
        "docs/figures/"
    ]
}

summary_df = pd.DataFrame(summary_data)

# Display with styling
display(HTML("<h3>📦 Project Data Summary</h3>"))
display(summary_df.style.set_properties(**{
    'background-color': '#f0f0f0',
    'color': 'black',
    'border-color': 'white'
}))

print("\n" + "="*60)
print("📋 Available Files:")
print("="*60)
if raw_files:
    print("\n📊 Raw Data:")
    for f in raw_files:
        print(f"  • {f.name}")
if processed_files:
    print("\n🔧 Processed Data:")
    for f in processed_files:
        print(f"  • {f.name}")
if model_files:
    print("\n🤖 Models:")
    for f in model_files:
        print(f"  • {f.name}")
if figure_files:
    print(f"\n📈 Visualizations: {len(figure_files)} figures available")

---
## 🤖 Model Information

In [ ]:
# Load and display model information
if model_files:
    display(HTML("<h3>🔍 Trained Models Details</h3>"))
    
    for model_file in model_files:
        print("\n" + "="*60)
        print(f"📦 Model: {model_file.name}")
        print("="*60)
        
        try:
            model = joblib.load(model_file)
            
            print(f"\n✅ Model Type: {type(model).__name__}")
            print(f"📊 Number of Features: {model.n_features_in_}")
            print(f"🎯 Classes: {model.classes_}")
            
            # Get parameters
            params = model.get_params()
            print("\n⚙️  Model Parameters:")
            for key, value in list(params.items())[:10]:  # Show first 10 params
                if value is not None and not callable(value):
                    print(f"   • {key}: {value}")
            
            # Feature importance if available
            if hasattr(model, 'coef_'):
                print("\n📈 Feature Coefficients:")
                feature_names = ['returns', 'ma_5', 'ma_20', 'volatility', 'price_to_ma']
                for i, coef in enumerate(model.coef_[0]):
                    print(f"   • {feature_names[i]}: {coef:.6f}")
            
            # File size
            file_size = model_file.stat().st_size / 1024  # KB
            print(f"\n💾 File Size: {file_size:.2f} KB")
            
        except Exception as e:
            print(f"❌ Error loading model: {e}")
else:
    print("⚠️  No trained models found. Run main.py or main_demo.py first.")

---
## 📊 Visualizations

All generated visualizations displayed in an organized grid.

In [ ]:
# Display all visualizations
if figure_files:
    display(HTML("<h3>🎨 Generated Visualizations</h3>"))
    
    # Group figures by ticker
    ticker_figures = {}
    other_figures = []
    
    for fig in figure_files:
        name = fig.stem
        if "AAPL" in name:
            ticker_figures.setdefault("AAPL", []).append(fig)
        elif "SPY" in name:
            ticker_figures.setdefault("SPY", []).append(fig)
        else:
            other_figures.append(fig)
    
    # Display figures by ticker
    for ticker, figs in sorted(ticker_figures.items()):
        display(HTML(f"<h4>📈 {ticker} Visualizations</h4>"))
        
        for fig in sorted(figs):
            display(HTML(f"<p><strong>{fig.stem}</strong></p>"))
            display(Image(filename=str(fig), width=800))
            display(HTML("<hr>"))
    
    # Display other figures
    if other_figures:
        display(HTML("<h4>📊 Comparison & Summary Visualizations</h4>"))
        for fig in sorted(other_figures):
            display(HTML(f"<p><strong>{fig.stem}</strong></p>"))
            display(Image(filename=str(fig), width=800))
            display(HTML("<hr>"))
else:
    print("⚠️  No visualizations found. Run main.py or main_demo.py first.")

---
## 📉 Performance Metrics

Load and analyze model performance from available data.

In [ ]:
# This section would load performance metrics if they were saved
# For now, we'll create a template

display(HTML("<h3>📊 Model Performance Summary</h3>"))

print("""
To view detailed performance metrics:

1. Run the pipeline: python main_demo.py
2. Check the console output for detailed metrics
3. Or load the processed data and predictions to calculate metrics

Key Metrics to Track:
  • Accuracy (test set)
  • Precision (positive class)
  • Recall (positive class)
  • F1 Score
  • AUC-ROC
  • Baseline Comparison
""")

# Example: Load processed data and calculate basic statistics
if processed_files:
    print("\n" + "="*60)
    print("📈 Data Statistics")
    print("="*60)
    
    for pfile in processed_files:
        print(f"\n📊 {pfile.name}:")
        df = pd.read_csv(pfile, index_col=0, parse_dates=True)
        print(f"  • Shape: {df.shape}")
        print(f"  • Date Range: {df.index.min().date()} to {df.index.max().date()}")
        print(f"  • Columns: {', '.join(df.columns[:8])}{'...' if len(df.columns) > 8 else ''}")
        
        if 'target' in df.columns:
            target_dist = df['target'].value_counts()
            print(f"  • Target Distribution:")
            print(f"      - Down days (0): {target_dist.get(0, 0)} ({target_dist.get(0, 0)/len(df)*100:.1f}%)")
            print(f"      - Up days (1): {target_dist.get(1, 0)} ({target_dist.get(1, 0)/len(df)*100:.1f}%)")

---
## 🔍 Raw Data Exploration

Interactive exploration of the raw and processed data.

In [ ]:
# Load and explore processed data
if processed_files:
    display(HTML("<h3>🔍 Interactive Data Exploration</h3>"))
    
    # Load first processed file as example
    example_file = processed_files[0]
    df = pd.read_csv(example_file, index_col=0, parse_dates=True)
    
    print(f"\n📊 Displaying: {example_file.name}\n")
    
    # Basic info
    display(HTML("<h4>📋 Data Info</h4>"))
    print(df.info())
    
    # First few rows
    display(HTML("<h4>👀 First 10 Rows</h4>"))
    display(df.head(10))
    
    # Last few rows
    display(HTML("<h4>🔚 Last 10 Rows</h4>"))
    display(df.tail(10))
    
    # Summary statistics
    display(HTML("<h4>📊 Summary Statistics</h4>"))
    display(df.describe().round(4))
    
    # Missing values
    display(HTML("<h4>❓ Missing Values</h4>"))
    missing = df.isnull().sum()
    if missing.sum() > 0:
        display(missing[missing > 0])
    else:
        print("✅ No missing values!")
    
    # Correlations
    if len(df.columns) > 1:
        display(HTML("<h4>🔗 Feature Correlations</h4>"))
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 1:
            plt.figure(figsize=(10, 8))
            sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, 
                       square=True, linewidths=1, cbar_kws={"shrink": 0.8})
            plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
else:
    print("⚠️  No processed data found. Run main.py or main_demo.py first.")

---
## 🎯 Custom Analysis

Use this section to add your own custom data exploration and analysis.

In [ ]:
# Add your custom analysis here
print("💡 This cell is ready for your custom analysis!")
print("\nExamples:")
print("  • Compare multiple models")
print("  • Analyze specific time periods")
print("  • Test different thresholds")
print("  • Create custom visualizations")
print("  • Calculate additional metrics")

---
## 📝 Quick Reference

### Useful Variables
- `project_root` - Path to project root directory
- `data_dir` - Path to data directory
- `models_dir` - Path to models directory
- `figures_dir` - Path to figures directory
- `df` - Last loaded DataFrame (from processed data)

### Quick Commands
```python
# Load a specific processed file
df = pd.read_csv(data_dir / "processed" / "AAPL_processed.csv", index_col=0, parse_dates=True)

# Load a model
model = joblib.load(models_dir / "AAPL_logistic.pkl")

# Display an image
display(Image(filename=str(figures_dir / "TEST_AAPL_predictions.png"), width=800))
```

---

**Last Updated**: 2026-02-12  
**Status**: Ready for interactive exploration  
**Next**: Run all cells to view your data!